# Blood–Brain Barrier Exclusion (BBBX) Score

Use the repository's Python environment and run the cells in order. In **Select input**, set `test_set` to your CSV or SDF, or choose SMILES and enter them directly.

Training-set checks add flags only; flagged compounds are still predicted. Invalid inputs remain in the output with an error. For numerical replay of earlier results, choose **Descriptor CSV** and use the saved processed dataset.


In [8]:
# Import packages

import base64
import hashlib
import io
import os
import shutil
import subprocess
import sys
import tempfile
from datetime import datetime
from pathlib import Path
import joblib
import lightgbm as lgb
import numpy as np
import pandas as pd
from IPython.display import display
from mordred import Calculator, descriptors
from rdkit import Chem, RDLogger
from rdkit.Chem import Crippen, Descriptors, Lipinski, MACCSkeys, rdMolDescriptors

RDLogger.DisableLog("rdApp.warning")


In [ ]:
# Select input and then run all cells after that. Predictions will appear after the 

input_kind = "CSV"  # "CSV", "SMILES", "3D SDF", or "Descriptor CSV"

file = "./bbbx_operation_test_set.csv" # or test "./bbbx_operation_test_set.sdf" or "./bbbx_operation_test_set_labelled.csv"
smiles_space_separated = "CC(=O)Nc1ccc(O)cc1"

mol_names_optional = ""  # Optional comma-separated names, one per SMILES.
smiles_column = "SMILES"
name_column = "Name"

In [ ]:
# Existing model and structure-preparation files; edit these paths if needed.
# Defaults work from the repository root or from final_model/.
PROJECT_ROOT = Path.cwd().parent
MODEL_DIR = PROJECT_ROOT / "final_model/models"
OPEN_CHEM_LIB_JAR = PROJECT_ROOT / "tools/datawarrior_all.jar"
OUTPUT_DIR = PROJECT_ROOT / "final_model"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [11]:
# Read input and check the training-set criteria

def training_set_flags(mol):
    # Match notebook 01, including rounding before the numerical comparisons.
    values = pd.Series({"MW": Descriptors.ExactMolWt(mol), "SLogP": Crippen.MolLogP(mol),
                        "TopoPSA": rdMolDescriptors.CalcTPSA(mol)}).round(2)
    checks = {
        "fewer than 2 heavy atoms": mol.GetNumHeavyAtoms() >= 2,
        "multiple disconnected fragments": len(Chem.GetMolFrags(mol)) == 1,
        "no carbon atoms": any(a.GetAtomicNum() == 6 for a in mol.GetAtoms()),
        "MW > 800": values["MW"] <= 800,
        "H-bond donors > 6": Lipinski.NumHDonors(mol) <= 6,
        "H-bond acceptors > 11": Lipinski.NumHAcceptors(mol) <= 11,
        "SLogP outside (0, 5]": 0 < values["SLogP"] <= 5,
        "TopoPSA >= 180": values["TopoPSA"] < 180,
    }
    return "; ".join(label for label, passed in checks.items() if not passed)

def connectivity(mol):
    copy = Chem.Mol(mol)
    Chem.RemoveStereochemistry(copy)
    return Chem.MolToSmiles(Chem.RemoveHs(copy), isomericSmiles=False)

input_file = Path(file).expanduser() if input_kind != "SMILES" else None
rows, supplied_molecules = [], {}
if input_kind == "SMILES":
    smiles = smiles_space_separated.split()
    names = [s.strip() for s in mol_names_optional.split(",")] if mol_names_optional.strip() else []
    if names and len(names) != len(smiles):
        raise ValueError("Provide one comma-separated name per SMILES, or leave names blank.")
    rows = [{"name": names[i] if names else f"compound_{i+1}", "smiles": smi, "error": ""}
            for i, smi in enumerate(smiles)]
elif input_kind in {"CSV", "Descriptor CSV"}:
    table = pd.read_csv(input_file, dtype=str, keep_default_na=False)
    if smiles_column not in table:
        raise ValueError(f"Missing SMILES column: {smiles_column}")
    rows = [{"name": row.get(name_column, "") or f"compound_{i+1}",
             "smiles": row[smiles_column].strip(), "error": ""} for i, row in table.iterrows()]
elif input_kind == "3D SDF":
    # Preserve explicit hydrogens, as in the original Mordred CLI workflow.
    for i, mol in enumerate(Chem.SDMolSupplier(str(input_file), removeHs=False)):
        row = {"name": f"compound_{i+1}", "smiles": "", "error": ""}
        if mol is None:
            row["error"] = "Invalid SDF record."
        else:
            row["name"] = mol.GetProp("_Name") if mol.HasProp("_Name") else row["name"]
            row["smiles"] = next((mol.GetProp(key).strip() for key in ("INPUT_SMILES", "SMILES")
                                   if mol.HasProp(key) and mol.GetProp(key).strip()), Chem.MolToSmiles(Chem.RemoveHs(mol)))
            parsed = Chem.MolFromSmiles(row["smiles"])
            if parsed is None or connectivity(parsed) != connectivity(mol):
                row["error"] = "SDF structure and its SMILES property do not match."
            elif not mol.GetNumConformers() or not mol.GetConformer().Is3D():
                row["error"] = "A 3D SDF is required; use SMILES/CSV to generate coordinates."
            elif not np.isfinite(mol.GetConformer().GetPositions()).all():
                row["error"] = "Nonfinite SDF coordinates."
            supplied_molecules[i+1] = mol
        rows.append(row)
else:
    raise ValueError("Choose SMILES, CSV, 3D SDF, or Descriptor CSV.")
if not rows:
    raise ValueError("No input compounds were provided.")
input_df = pd.DataFrame(rows)
input_df.index = pd.Index(range(1, len(input_df)+1), name="input_row")
print(f"Read {len(input_df)} compounds.")

input_df["status"] = "ready"
input_df["training_set_flags"] = ""
input_df["matches_training_criteria"] = pd.Series(pd.NA, index=input_df.index, dtype="boolean")
for row_id, row in input_df.iterrows():
    mol = Chem.MolFromSmiles(row["smiles"]) if row["smiles"] else None
    if row["error"] or mol is None or mol.GetNumAtoms() == 0:
        input_df.loc[row_id, "status"] = "invalid_input"
        input_df.loc[row_id, "error"] = row["error"] or "Invalid or empty SMILES."
    else:
        flags = training_set_flags(mol)
        input_df.loc[row_id, "training_set_flags"] = flags
        input_df.loc[row_id, "matches_training_criteria"] = not bool(flags)

# Training-set flags do NOT remove compounds from the prediction list.
valid_ids = input_df.index[input_df["status"].eq("ready")].tolist()

flagged = input_df["training_set_flags"].ne("").sum()
print(f"{flagged} compound(s) flagged against the training criteria.")


Read 1 compounds.
0 compound(s) flagged against the training criteria.


In [12]:
# Generate 3D conformers (SMILES/CSV inputs only)

BASE_SEED = 20260915
DIELECTRIC_CONSTANT = 1

def find_java():
    candidates = [Path(sys.prefix) / "bin/java"]
    if os.environ.get("JAVA_HOME"):
        candidates.append(Path(os.environ["JAVA_HOME"]) / "bin/java")
    if shutil.which("java"):
        candidates.append(Path(shutil.which("java")))
    for candidate in candidates:
        if candidate.is_file():
            check = subprocess.run([str(candidate), "--list-modules"], capture_output=True, text=True)
            if check.returncode == 0 and "jdk.compiler@" in check.stdout:
                return str(candidate.resolve())
    return None

# Unchanged seeded OpenChemLib helper from notebook 01.
JAVA_SOURCE = r'''
import com.actelion.research.chem.MolfileV3Creator;
import com.actelion.research.chem.StereoMolecule;
import com.actelion.research.chem.SmilesParser;
import com.actelion.research.chem.forcefield.mmff.ForceFieldMMFF94;
import org.openmolecules.chem.conf.gen.ConformerGenerator;
import org.openmolecules.chem.conf.gen.RigidFragmentCache;
import java.io.*;
import java.nio.charset.StandardCharsets;
import java.nio.file.*;
import java.util.*;

public class SeededDataWarriorConformers {
  static final int MAX_TORSION_SETS = 1000;
  static final int MAX_ITERATIONS = 10000;
  static final double GRADIENT_TOLERANCE = 0.0001;
  static final double FUNCTION_TOLERANCE = 1.0e-6;

  public static void main(String[] args) throws Exception {
    Path manifest = Path.of(args[0]);
    Path output = Path.of(args[1]);
    Path failures = Path.of(args[2]);
    double dielectric = Double.parseDouble(args[3]);
    RigidFragmentCache.getDefaultInstance().loadDefaultCache();
    ForceFieldMMFF94.initialize(ForceFieldMMFF94.MMFF94SPLUS);
    Map<String,Object> options = new HashMap<>();
    options.put("dielectric constant", dielectric);
    int inputCount = 0, writtenCount = 0, failureCount = 0;

    try (BufferedReader in = Files.newBufferedReader(manifest, StandardCharsets.UTF_8);
         BufferedWriter sdf = Files.newBufferedWriter(output, StandardCharsets.UTF_8);
         BufferedWriter failed = Files.newBufferedWriter(failures, StandardCharsets.UTF_8)) {
      failed.write("dataframe_row\tinput_id\tsmiles\terror\n");
      String line;
      while ((line = in.readLine()) != null) {
        inputCount++;
        String[] f = line.split("\\t", -1);
        String row = decode(f[0]), id = decode(f[1]), smiles = decode(f[2]);
        long seed = Long.parseLong(f[3]);
        String originalProperties = decode(f[4]);
        try {
          StereoMolecule mol = new StereoMolecule();
          new SmilesParser().parse(mol, smiles);
          ConformerGenerator cg = new ConformerGenerator(seed, true);
          if (!cg.initializeConformers(mol, ConformerGenerator.STRATEGY_LIKELY_SYSTEMATIC,
                  MAX_TORSION_SETS, false))
            throw new Exception("Conformer generator could not initialize");
          mol = cg.getNextConformerAsMolecule(mol);
          if (mol == null) throw new Exception("No conformer generated");
          double energy = Double.NaN;
          String minimizationError = "";

          try {
            energy = minimize(mol, options);
          } catch (Exception e) {
            // Match DataWarrior: retain the generated conformer when
            // MMFF94 typing or minimization fails.
            minimizationError = e.getClass().getSimpleName() + ": " + e.getMessage();
          }

          center(mol);
          mol.setName(id.isBlank() ? "row " + row : id);
          sdf.write(new MolfileV3Creator(mol).getMolfile());
          // Restore every original CSV column using its exact original name.
          for (String entry : originalProperties.split("\u001e", -1)) {
            if (entry.isEmpty()) continue;

            int separator = entry.indexOf('\u001f');
            if (separator < 0) continue;

            String columnName = entry.substring(0, separator);
            String columnValue = entry.substring(separator + 1);

            property(sdf, columnName, columnValue);
          }
          property(sdf, "DATAFRAME_ROW", row);
          property(sdf, "INPUT_ID", id);
          property(sdf, "INPUT_SMILES", smiles);
          property(sdf, "CONFORMER_SEED", Long.toString(seed));
          property(sdf, "CONFORMER_ALGORITHM", "systematic, low-energy biased");
          property(sdf, "TORSION_SOURCE", "crystallographic");
          property(sdf, "FORCE_FIELD", "MMFF94s+");
          property(sdf, "DIELECTRIC_CONSTANT", Double.toString(dielectric));
          property(
              sdf,
              "MMFF94_ENERGY",
              Double.isNaN(energy) ? "" : String.format(Locale.ROOT, "%.9g", energy)
          );
          property(sdf, "MINIMIZATION_ERROR", minimizationError);
          sdf.write("$$$$\n");
          writtenCount++;
        } catch (Throwable e) {
          failureCount++;
          failed.write(clean(row)+"\t"+clean(id)+"\t"+clean(smiles)+"\t"
              +clean(e.getClass().getSimpleName()+": "+e.getMessage())+"\n");
        }
      }
    }
    System.out.printf(Locale.ROOT, "Input: %d  Written: %d  Failed: %d%n",
        inputCount, writtenCount, failureCount);
    if (failureCount != 0) System.exit(1);
  }

  static double minimize(StereoMolecule mol, Map<String,Object> options) throws Exception {
    int[] fragmentNo = new int[mol.getAllAtoms()];
    int fragmentCount = mol.getFragmentNumbers(fragmentNo, false, true);
    double energy = Double.NaN;
    if (fragmentCount == 1) {
      ForceFieldMMFF94 ff = new ForceFieldMMFF94(mol, ForceFieldMMFF94.MMFF94SPLUS, options);
      int error = ff.minimise(MAX_ITERATIONS, GRADIENT_TOLERANCE, FUNCTION_TOLERANCE);
      if (error != 0) throw new Exception("MMFF94 error code " + error);
      energy = ff.getTotalEnergy();
    } else {
      StereoMolecule[] fragments = mol.getFragments(fragmentNo, fragmentCount);
      int maxAtoms = 0;
      for (StereoMolecule fragment : fragments) {
        if (fragment.getAllAtoms() > 2) {
          ForceFieldMMFF94 ff = new ForceFieldMMFF94(fragment, ForceFieldMMFF94.MMFF94SPLUS, options);
          int error = ff.minimise(MAX_ITERATIONS, GRADIENT_TOLERANCE, FUNCTION_TOLERANCE);
          if (error != 0) throw new Exception("MMFF94 error code " + error);
          if (maxAtoms < fragment.getAllAtoms()) {
            maxAtoms = fragment.getAllAtoms(); energy = ff.getTotalEnergy();
          }
        }
      }
      int[] atomInFragment = new int[fragmentCount];
      for (int atom=0; atom<fragmentNo.length; atom++) {
        int fragment = fragmentNo[atom], sourceAtom = atomInFragment[fragment]++;
        mol.setAtomX(atom, fragments[fragment].getAtomX(sourceAtom));
        mol.setAtomY(atom, fragments[fragment].getAtomY(sourceAtom));
        mol.setAtomZ(atom, fragments[fragment].getAtomZ(sourceAtom));
      }
    }
    return energy;
  }

  static void center(StereoMolecule mol) {
    double x=0, y=0, z=0;
    for (int a=0; a<mol.getAllAtoms(); a++) { x+=mol.getAtomX(a); y+=mol.getAtomY(a); z+=mol.getAtomZ(a); }
    x/=mol.getAllAtoms(); y/=mol.getAllAtoms(); z/=mol.getAllAtoms();
    for (int a=0; a<mol.getAllAtoms(); a++) {
      mol.setAtomX(a,mol.getAtomX(a)-x); mol.setAtomY(a,mol.getAtomY(a)-y); mol.setAtomZ(a,mol.getAtomZ(a)-z);
    }
  }

  static void property(BufferedWriter w, String name, String value) throws Exception {
    w.write(">  <"+name+">\n"+clean(value)+"\n\n");
  }
  static String decode(String s) { return new String(Base64.getDecoder().decode(s), StandardCharsets.UTF_8); }
  static String clean(String s) { return s == null ? "" : s.replace('\t',' ').replace('\r',' ').replace('\n',' '); }
}
'''

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
molecules = {i: supplied_molecules[i] for i in valid_ids} if input_kind == "3D SDF" else {}
if input_kind in {"SMILES", "CSV"} and valid_ids:
    java = find_java()
    if java is None:
        raise RuntimeError("A Java JDK is required for SMILES/CSV input. Activate the repository environment, or use a prepared 3D SDF/descriptor CSV.")
    jar = Path(OPEN_CHEM_LIB_JAR).expanduser().resolve()
    if not jar.is_file():
        raise FileNotFoundError("Set OPEN_CHEM_LIB_JAR to the DataWarrior JAR used in the dataset workflow.")
    sdf_path = OUTPUT_DIR / f"prepared_3d_{timestamp}.sdf"
    failure_path = OUTPUT_DIR / f"conformer_failures_{timestamp}.tsv"
    encode = lambda value: base64.b64encode(str(value).encode()).decode()
    with tempfile.TemporaryDirectory(prefix="bbbx_3d_") as temporary:
        temporary = Path(temporary)
        source = temporary / "SeededDataWarriorConformers.java"
        source.write_text(JAVA_SOURCE, encoding="utf-8")
        manifest = temporary / "input.tsv"
        with manifest.open("w", encoding="utf-8") as handle:
            for row_id in valid_ids:
                row = input_df.loc[row_id]
                seed = int.from_bytes(hashlib.sha256(f"{BASE_SEED}\0{row['smiles']}".encode()).digest()[:8], "big", signed=True) or 1
                properties = f"INPUT_NAME\x1f{row['name']}\x1eSMILES\x1f{row['smiles']}"
                handle.write("\t".join([encode(row_id), encode(row_id), encode(row["smiles"]), str(seed), encode(properties)]) + "\n")
        result = subprocess.run([java, "--class-path", str(jar), str(source), str(manifest),
                                 str(sdf_path), str(failure_path), str(DIELECTRIC_CONSTANT)], capture_output=True, text=True)
    if result.returncode not in (0, 1) or not failure_path.is_file():
        raise RuntimeError(result.stderr or "Conformer generation could not run.")
    failures = pd.read_csv(failure_path, sep="\t", dtype=str, keep_default_na=False)
    for _, failure in failures.iterrows():
        row_id = int(failure["dataframe_row"])
        input_df.loc[row_id, ["status", "error"]] = ["preparation_failed", failure["error"]]
    if sdf_path.is_file() and sdf_path.stat().st_size:
        for mol in Chem.SDMolSupplier(str(sdf_path), removeHs=False):
            if mol is None or not mol.HasProp("DATAFRAME_ROW"):
                raise RuntimeError("A generated SDF record lost its input-row identifier.")
            row_id = int(mol.GetProp("DATAFRAME_ROW"))
            if row_id not in valid_ids or row_id in molecules or input_df.loc[row_id, "status"] != "ready":
                raise RuntimeError("Generated structures do not align with input rows.")
            if mol.GetProp("INPUT_SMILES") != input_df.loc[row_id, "smiles"]:
                raise RuntimeError("Generated SMILES do not align with input rows.")
            molecules[row_id] = mol
    if set(molecules) | set(failures["dataframe_row"].astype(int)) != set(valid_ids):
        raise RuntimeError("Some conformer outcomes are unaccounted for.")
    print(result.stdout.strip())


Input: 1  Written: 1  Failed: 0


In [13]:
# Calculate Descriptors

# Load each fold's saved feature order; do not select columns from the input batch.
models = []
for model_id in range(5):
    saved = joblib.load(MODEL_DIR / f"custom_mtl_bab_outer_{model_id}_preprocessor.joblib")
    names = list(saved["feature_names"])
    if saved.get("missing_value_policy") != "native_lightgbm_nan_handling_no_imputation":
        raise ValueError(f"Fold {model_id}: use the updated model/preprocessor pair.")
    if float(saved.get("threshold", -1)) != -1 or len(names) != len(set(names)):
        raise ValueError(f"Fold {model_id}: unexpected model metadata.")
    model = lgb.Booster(model_file=str(MODEL_DIR / f"custom_mtl_bab_outer_{model_id}.txt"))
    if model.num_feature() != len(names):
        raise ValueError(f"Fold {model_id}: model and feature list do not match.")
    models.append((model, names))
feature_names = list(dict.fromkeys(name for _, names in models for name in names))

if input_kind == "Descriptor CSV":
    # Preserve the default pandas CSV parser used by prediction script 11.
    supplied = pd.read_csv(input_file, low_memory=False)
    if len(supplied) != len(input_df):
        raise ValueError("Input and descriptor row counts differ.")
    missing = [name for name in feature_names if name not in supplied]
    if missing:
        raise ValueError(f"Missing {len(missing)} required descriptors; first missing: {missing[:10]}")
    supplied.index = input_df.index
    fully_labelled_df = supplied.loc[valid_ids, feature_names].apply(pd.to_numeric, errors="coerce")
else:
    registry = {str(d): d for d in Calculator(descriptors, ignore_3D=False).descriptors}
    mordred_names = [name for name in feature_names if name in registry]
    fp_names = {f"ECFP_{i}" for i in range(2048)} | {f"MACCS_{i}" for i in range(167)}
    if set(feature_names) - set(mordred_names) - fp_names:
        raise ValueError("The saved feature list contains an unknown descriptor.")
    calculator = Calculator([registry[name] for name in mordred_names], ignore_3D=False)
    feature_rows = []
    for row_id, mol in molecules.items():
        two_d = Chem.MolFromSmiles(input_df.loc[row_id, "smiles"])
        ecfp = rdMolDescriptors.GetMorganFingerprintAsBitVect(two_d, 2, nBits=2048, useChirality=False, useFeatures=False)
        maccs = MACCSkeys.GenMACCSKeys(two_d)
        values = {f"ECFP_{i}": int(ecfp[i]) for i in range(2048)}
        values.update({f"MACCS_{i}": int(maccs[i]) for i in range(167)})
        for name, value in zip(mordred_names, calculator(mol)):
            try:
                number = float(value)
            except (ValueError, TypeError):
                number = np.nan
            values[name] = (str(int(value)) if isinstance(value, (bool, np.bool_)) else str(value)) if np.isfinite(number) else ""
        feature_rows.append(pd.Series(values, name=row_id))
    fully_labelled_df = pd.DataFrame(feature_rows, columns=feature_names)
    fully_labelled_df.index.name = "input_row"
    # Match the Mordred CSV read, then assembled-dataset CSV read in the original workflow.
    if len(fully_labelled_df):
        for _ in range(2):
            fully_labelled_df = pd.read_csv(io.StringIO(fully_labelled_df.to_csv()), index_col="input_row", low_memory=False)
    fully_labelled_df = fully_labelled_df.apply(pd.to_numeric, errors="coerce")

fully_labelled_df = fully_labelled_df.replace([np.inf, -np.inf], np.nan)
fully_labelled_df.index.name = "input_row"
descriptor_file = OUTPUT_DIR / f"fully_labelled_dataset_{timestamp}.csv"
metadata = input_df.loc[fully_labelled_df.index, ["name", "smiles"]].rename(columns={"name": "Name", "smiles": "SMILES"})
metadata.join(fully_labelled_df).to_csv(descriptor_file)
print(f"Descriptors saved to: {descriptor_file}")


Descriptors saved to: /Users/sziadat/Documents/bbbx/final_package/bbb-exc-score/final_model/fully_labelled_dataset_20261101_142902_012457.csv


In [14]:
# Get Score Values

output_df = input_df.copy()
output_df["predicted_logBB"] = np.nan
output_df["BBBX_score"] = np.nan
output_df["predicted_BBB_label"] = ""
all_predictions = []
if len(fully_labelled_df):
    for model, names in models:
        # Missing descriptor values stay NaN; LightGBM handles them natively.
        X_all = fully_labelled_df.loc[:, names].to_numpy(dtype=float)
        all_predictions.append(model.predict(X_all, num_threads=1))
    all_preds = np.mean(np.asarray(all_predictions), axis=0)
    if not np.isfinite(all_preds).all():
        raise RuntimeError("The model returned a nonfinite prediction.")
    predicted_ids = fully_labelled_df.index
    output_df.loc[predicted_ids, "predicted_logBB"] = all_preds
    output_df.loc[predicted_ids, "BBBX_score"] = (all_preds + 2.5) / 4.5 * 6
    output_df.loc[predicted_ids, "predicted_BBB_label"] = np.where(all_preds < -1, "nonpenetrant", "penetrant")
    output_df.loc[predicted_ids, "status"] = "predicted"

output_df = output_df.reset_index()[["input_row", "name", "smiles", "predicted_logBB", "BBBX_score",
    "predicted_BBB_label", "matches_training_criteria", "training_set_flags", "status", "error"]]
unique_file_name = OUTPUT_DIR / f"predicted_scores_{timestamp}.csv"
output_df.to_csv(unique_file_name, index=False)
print(f"Predictions saved to: {unique_file_name}")


Predictions saved to: /Users/sziadat/Documents/bbbx/final_package/bbb-exc-score/final_model/predicted_scores_20261101_142902_012457.csv


In [15]:
# Display Output
display(output_df)


,input_row,name,smiles,predicted_logBB,BBBX_score,predicted_BBB_label,matches_training_criteria,training_set_flags,status,error
0,1,compound_1,CC(=O)Nc1ccc(O)cc1,-0.356912,2.85745,penetrant,True,,predicted,
